# Olist E-commerce: Revenue Analysis with JOINs

**Questions:** How much revenue does Olist make and how is it growing? Which categories
and states drive it? Do big orders bring most of the money?

**Rules:** delivered orders only, January 2017 – August 2018 (complete months).
Revenue = sum of item prices (excluding shipping), in Brazilian Real (R$).

In [1]:
import duckdb

%load_ext sql
con = duckdb.connect("../data/olist.duckdb", read_only=True)
%sql con
%config SqlMagic.displaylimit = 50

In [2]:
%%sql
SELECT COUNT(*) AS rows_after_join,
       COUNT(DISTINCT o.order_id) AS real_orders
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01';

Running query in 'DuckDBPyConnection'

rows_after_join,real_orders
109880,96211


In [3]:
%%sql
SELECT COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(i.price), 0) AS revenue,
       ROUND(SUM(i.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01';

Running query in 'DuckDBPyConnection'

orders,revenue,avg_order_value
96211,13181027.0,137.0


In [4]:
%%sql
SELECT date_trunc('month', o.order_purchase_timestamp)::DATE AS month,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(i.price), 0) AS revenue,
       ROUND(SUM(i.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'
GROUP BY month
ORDER BY month;

Running query in 'DuckDBPyConnection'

month,orders,revenue,avg_order_value
2017-01-01,750,111798.0,149.06
2017-02-01,1653,234223.0,141.7
2017-03-01,2546,359199.0,141.08
2017-04-01,2303,340670.0,147.92
2017-05-01,3546,489338.0,138.0
2017-06-01,3135,421923.0,134.58
2017-07-01,3872,481605.0,124.38
2017-08-01,4193,554700.0,132.29
2017-09-01,4150,607400.0,146.36
2017-10-01,4478,648248.0,144.76


In [5]:
%%sql
SELECT COUNT(*) AS rows_inner_join
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
JOIN products p ON i.product_id = p.product_id
JOIN category_translation t ON p.product_category_name = t.product_category_name
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01';

Running query in 'DuckDBPyConnection'

rows_inner_join
108323


In [6]:
%%sql
SELECT COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(i.price), 0) AS revenue
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
LEFT JOIN products p ON i.product_id = p.product_id
LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;

Running query in 'DuckDBPyConnection'

category,orders,revenue
health_beauty,8610,1229557.0
watches_gifts,5491,1163466.0
bed_bath_table,9267,1022956.0
sports_leisure,7513,952840.0
computers_accessories,6518,888056.0
furniture_decor,6258,706237.0
housewares,5734,614342.0
cool_stuff,3552,609158.0
auto,3802,577838.0
garden_tools,3443,469135.0


In [7]:
%%sql
SELECT c.customer_state AS state,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(i.price), 0) AS revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'
GROUP BY state
ORDER BY revenue DESC
LIMIT 5;

Running query in 'DuckDBPyConnection'

state,orders,revenue
SP,40406,5055587.0
RJ,12310,1751434.0
MG,11319,1548207.0
RS,5328,726374.0
PR,4903,664048.0


In [8]:
%%sql
SELECT order_size,
       COUNT(*) AS orders,
       ROUND(SUM(order_value), 0) AS revenue
FROM (
    SELECT o.order_id,
           SUM(i.price) AS order_value,
           CASE WHEN SUM(i.price) < 50  THEN '1. Under R$50'
                WHEN SUM(i.price) < 100 THEN '2. R$50-100'
                WHEN SUM(i.price) < 200 THEN '3. R$100-200'
                ELSE '4. R$200+'
           END AS order_size
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY o.order_id
)
GROUP BY order_size
ORDER BY order_size;

Running query in 'DuckDBPyConnection'

order_size,orders,revenue
1. Under R$50,28709,913796.0
2. R$50-100,27654,2086385.0
3. R$100-200,25256,3628775.0
4. R$200+,14592,6552071.0


In [9]:
con.close()

## Findings: revenue (Jan 2017 – Aug 2018, delivered orders)

- **Total:** 96,211 orders, **R$ 13.18M revenue**, average order value **R$ 137**.
- **Growth:** monthly revenue grew ~8× during 2017, peaking at **Black Friday (Nov 2017, +63% vs October)**.
  In 2018 it **plateaued** at R$ 840K–980K per month. Growth came from more orders, not bigger orders.
- **Categories:** health & beauty (R$ 1.23M) and watches & gifts (R$ 1.16M) lead. Bed/bath/table has the
  most orders but cheaper items, so "most popular" ≠ "most valuable."
- **Geography:** **São Paulo = 38% of revenue.** High dependence on one state; other regions are an opportunity.
- **Order size:** orders above R$200 are **15% of orders but 50% of revenue.**

## SQL lessons
- After joining to `order_items`, count `DISTINCT order_id` (fan-out: 109,880 rows vs 96,211 orders).
- An INNER JOIN to products/categories silently dropped **1,557 items**. LEFT JOIN + COALESCE keeps them.